[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/081.%20The%203D%20Container_Truck%20Loading%20Problem%20%28Bin%20Packing%29/P81-Tier-11_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# P81-Tier-11: Physical-Cyber Synthesis (Programmable Matter)

## Problem context: 3D container loading

The three-dimensional container loading problem asks how to place rectangular items inside a rectangular container without overlap or boundary violations. Practical loading decisions can also involve weight limits, support, orientation, fragility, and loading order. This notebook presents a small, illustrative greedy packing simulation in which an item may be compressed before placement.

## Purpose, inputs, and outputs

The input is a fixed container and a list of items with dimensions, mass, and a programmability flag. The program processes the items in decreasing order of their current volume. For each item, it searches candidate locations on a regular 5 cm grid, rejects locations that violate the modeled constraints, and selects the feasible location with the highest placement score. A simple compression rule can change the dimensions of programmable items when estimated remaining volume is tight.

The output includes the placements found by this heuristic, unplaced items, occupied-volume utilization, shape-change and energy estimates, and visual summaries. The method is a greedy grid-search heuristic; it does not prove that a packing is globally optimal.

## Model assumptions and scope

- Items and the usable container interior are axis-aligned rectangular boxes. Rotation is not considered.
- Candidate positions are sampled at 0.05 m intervals. Feasibility is checked using exact axis-aligned bounding-box overlap tests at those sampled positions.
- Compression scales each dimension by the same factor. It preserves item mass; it does not model material stress, deformation limits beyond the configured ratio, or changes in item shape beyond the rectangular approximation.
- The total mass of placed items may not exceed the container's mass capacity. Support, stacking strength, load balance, fragility, wall movement, and loading or unloading order are not modeled.
- Energy is an illustrative estimate based on the configured volumetric power density and an estimated deformation duration. It is not a measured or validated actuator-energy model.
- Programmable matter, distributed control, and physical actuation are conceptual context. The code simulates a centralized computation and does not implement real programmable materials or autonomous distributed coordination.

## Tier 11 focus: physical-cyber synthesis

Physical-cyber systems connect computation with physical sensing and actuation. Programmable matter is a research concept in which material or modular structures can change their configuration under control. In logistics, a conceptual application would be to adapt an item's shape or a container's compartments to a loading plan.

The simulation below represents only a narrow part of that idea: a programmable item can be uniformly compressed before a greedy placement search. It does not demonstrate self-organization, real-time feedback, physical reconfiguration, or experimentally validated improvements. Those capabilities would require physical hardware, safety constraints, sensing, and control models beyond this packing example.

### Concepts and engineering considerations

- **Programmable matter:** materials or assemblies whose configuration can be changed through controlled actuation.
- **Physical-cyber system:** a system that connects computation with physical sensing and actuation.
- **Dynamic reconfiguration:** changing a physical configuration in response to a control decision.
- **Distributed intelligence:** computation distributed among system components rather than performed only by one central controller.

Potential application areas include instrumented warehouses, configurable storage systems, and research into adaptive cargo handling. Possible engineering challenges include energy use, reliability, safety, maintenance, and the practical limits of material deformation. These are considerations, not performance claims established by this notebook.

In [ ]:
import random
import time
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

warnings.filterwarnings("ignore")

# Fixed seeds keep any future stochastic extensions reproducible; this heuristic is deterministic.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully.")

## 1. Physical-cyber system representation

The classes in this section store the dimensions and properties used by the simulation. A **data class** is a compact way to define an object whose fields describe its state. The item dimensions are measured in metres, mass in kilograms, power density in watts per cubic metre, and deformation rate in metres per second.

The container's internal dimensions are its external dimensions minus the wall thickness on both sides of each axis. Item mass remains constant when dimensions change; the simulation does not calculate mass from a fixed density.

In [ ]:
@dataclass
class ProgrammableMatterProperties:
    """Illustrative physical parameters for a programmable component."""

    min_density: float = 100.0  # kg/m^3; recorded as a material property, not used in packing
    max_density: float = 2000.0  # kg/m^3; recorded as a material property, not used in packing
    shape_change_rate: float = 0.1  # m/s
    energy_consumption: float = 50.0  # W/m^3; used for an illustrative power estimate
    precision_level: float = 0.001  # m; recorded, while search positions use a 0.05 m grid
    response_time: float = 0.5  # s; recorded but not included in the energy estimate
    durability_cycles: int = 1000000


@dataclass
class SmartItem:
    """Rectangular cargo item, optionally allowed to use the compression rule."""

    id: int
    base_dimensions: Tuple[float, float, float]
    base_mass: float
    programmable: bool = True
    matter_properties: ProgrammableMatterProperties = field(
        default_factory=ProgrammableMatterProperties
    )
    current_shape: str = "rectangular"
    current_density: float = 500.0  # kg/m^3; descriptive and not used to infer mass
    deformation_capability: float = 0.3

    def get_current_dimensions(self) -> Tuple[float, float, float]:
        """Return the dimensions associated with the current shape state."""
        if self.current_shape == "compressed":
            factor = 1.0 - self.deformation_capability
            return tuple(dimension * factor for dimension in self.base_dimensions)
        if self.current_shape == "expanded":
            return tuple(dimension * 1.2 for dimension in self.base_dimensions)
        return self.base_dimensions

    def get_current_volume(self) -> float:
        """Calculate rectangular volume in cubic metres."""
        length, width, height = self.get_current_dimensions()
        return length * width * height

    def get_current_mass(self) -> float:
        """Return mass in kilograms; compression does not change item mass."""
        return self.base_mass


@dataclass
class SmartContainer:
    """Rectangular container with a fixed usable interior and mass limit."""

    id: int
    base_dimensions: Tuple[float, float, float]
    max_mass_capacity: float
    programmable: bool = True
    matter_properties: ProgrammableMatterProperties = field(
        default_factory=ProgrammableMatterProperties
    )
    current_internal_volume: float = 0.0
    compartment_config: str = "single"
    wall_thickness: float = 0.1

    def get_internal_dimensions(self) -> Tuple[float, float, float]:
        """Return usable interior dimensions after accounting for both walls."""
        dimensions = tuple(
            dimension - 2 * self.wall_thickness
            for dimension in self.base_dimensions
        )
        if any(dimension <= 0 for dimension in dimensions):
            raise ValueError("Wall thickness must leave positive interior dimensions.")
        return dimensions

    def get_internal_volume(self) -> float:
        """Calculate usable rectangular interior volume in cubic metres."""
        length, width, height = self.get_internal_dimensions()
        return length * width * height

    def reconfigure_compartment(self, new_config: str) -> bool:
        """Record a compartment configuration if the container is programmable."""
        if self.programmable:
            self.compartment_config = new_config
            return True
        return False

print("Container and item representations are defined.")

## 2. Greedy programmable-matter packing heuristic

### Intuitive purpose

A placement decision has two stages: first, the heuristic may compress an item when its estimated remaining volume is tight; next, it searches for a feasible position and chooses the one with the highest score. Processing larger items first is a common greedy rule: large items can be harder to place after smaller items occupy space. This rule is useful for a simple demonstration, but it does not guarantee the best packing.

### Formal model

Let the usable container dimensions be \((L,W,H)\), and let item \(i\) have dimensions \((l_i,w_i,h_i)\), mass \(m_i\), and position \(p_i=(x_i,y_i,z_i)\). The item occupies the box

$$
[x_i,x_i+l_i]\times[y_i,y_i+w_i]\times[z_i,z_i+h_i].
$$

The item must remain within the container:

$$
0\leq x_i\leq L-l_i,\qquad
0\leq y_i\leq W-w_i,\qquad
0\leq z_i\leq H-h_i.
$$

For every item already placed, the candidate box must be separated from it along at least one axis. For candidate item \(i\) and placed item \(j\), this non-overlap condition is

$$
x_i+l_i\leq x_j\quad\text{or}\quad x_j+l_j\leq x_i\quad\text{or}\quad
y_i+w_i\leq y_j\quad\text{or}\quad y_j+w_j\leq y_i\quad\text{or}\quad
z_i+h_i\leq z_j\quad\text{or}\quad z_j+h_j\leq z_i.
$$

Touching faces are allowed because the separation inequalities are non-strict. The total mass of the candidate and the items already placed must not exceed the container capacity. The implementation checks these conditions with axis-aligned bounding boxes and a cumulative mass calculation.

For a feasible position, the heuristic maximizes the score

$$
F_i(x_i,y_i,z_i)=0.4G_i+0.3C_i+0.3S_i,
$$
where

$$
G_i=\frac{1}{1+z_i},\qquad
C_i=\frac{1}{1+x_i+y_i},\qquad
S_i=\frac{l_iw_i}{LW}.
$$

Here, \(G_i\) rewards lower positions, \(C_i\) rewards positions near the origin corner, and \(S_i\) rewards a large base area relative to the container floor. The score is a heuristic preference, not a complete physical stability model. Coordinates are searched on a grid with spacing \(\Delta=0.05\) m. Thus, for each item in the chosen order, the method selects the feasible sampled position with the largest \(F_i\); if the sampled candidate set contains no feasible position, that item is left unplaced.

For compression, the heuristic compares the estimated remaining volume \(R\) with 1.5 times the item's current volume \(V_i\). If \(R<1.5V_i\), and the item is programmable, each dimension is scaled by \(1-d_i\), where \(d_i\) is its deformation capability. The item mass is unchanged. The estimated energy for this shape change is

$$
E_i = \rho_i V_i^{\mathrm{compressed}} t_i,
\qquad
 t_i=\frac{d_i\max(l_i,w_i,h_i)}{v_i}.
$$

In this estimate, \(\rho_i\) is the configured power density in W/m\(^3\), \(V_i^{\mathrm{compressed}}\) is the compressed volume in m\(^3\), and \(v_i\) is the configured deformation rate in m/s. Thus \(E_i\) is in joules. This is an illustrative model, not a measured actuator model.

### Algorithm and complexity

Items are prepared for compression, sorted by current volume in descending order, and considered one at a time. For each item, the algorithm enumerates grid positions, checks feasibility against already placed items, and retains the feasible position with the largest score. If there is no feasible sampled position, the item remains unplaced. There is no dynamic-programming state or Bellman recurrence: the method is a sequential greedy search.

If each container axis has approximately \(n\) candidate coordinates and there are \(k\) items, the search evaluates on the order of \(k n^3\) candidate positions. Each feasibility check can compare against as many as \(k\) placed items, giving a straightforward upper bound of order \(k^2 n^3\). The actual number of candidates depends on item and container dimensions.

In [ ]:
class ProgrammableMatterOptimizer:
    """Greedy grid-search heuristic for rectangular container loading."""

    def __init__(self, container: SmartContainer, items: List[SmartItem]):
        self.container = container
        self.items = items
        self.placed_items = []
        self.unplaced_items = items.copy()
        self.optimization_history = []
        self.energy_consumed = 0.0
        self.shape_changes = 0

    def analyze_space_requirements(self) -> Dict[str, float]:
        """Summarize initial volume and the potential from allowed compression."""
        total_base_volume = sum(item.get_current_volume() for item in self.items)
        container_volume = self.container.get_internal_volume()
        compression_savings = sum(
            item.get_current_volume()
            * (1.0 - (1.0 - item.deformation_capability) ** 3)
            for item in self.items
            if item.programmable
        )
        return {
            "total_item_volume": total_base_volume,
            "container_volume": container_volume,
            "space_utilization": total_base_volume / container_volume,
            "compression_potential": compression_savings,
            "optimized_utilization": (
                total_base_volume - compression_savings
            ) / container_volume,
        }

    def optimize_item_shapes(self) -> List[SmartItem]:
        """Apply the volume-based compression rule in the input item order."""
        optimized_items = []
        estimated_remaining_volume = self.container.get_internal_volume() - sum(
            item.get_current_volume() for item, _ in self.placed_items
        )

        for item in self.unplaced_items:
            current_volume = item.get_current_volume()
            if (
                item.programmable
                and estimated_remaining_volume < current_volume * 1.5
            ):
                item.current_shape = "compressed"
                self.shape_changes += 1
                properties = item.matter_properties
                deformation_distance = (
                    item.deformation_capability * max(item.base_dimensions)
                )
                duration = deformation_distance / properties.shape_change_rate
                power = properties.energy_consumption * item.get_current_volume()
                self.energy_consumed += power * duration

            optimized_items.append(item)
            estimated_remaining_volume -= item.get_current_volume()

        return optimized_items

    def find_optimal_position(
        self, item: SmartItem
    ) -> Optional[Tuple[float, float, float]]:
        """Return the highest-scoring feasible point on the search grid."""
        item_dimensions = item.get_current_dimensions()
        container_dimensions = self.container.get_internal_dimensions()
        step = 0.05
        best_position = None
        best_fitness = -float("inf")

        upper_limits = [
            container_dimensions[axis] - item_dimensions[axis]
            for axis in range(3)
        ]
        if any(limit < 0 for limit in upper_limits):
            return None

        coordinates = [
            np.arange(0.0, limit + step * 0.5, step)
            for limit in upper_limits
        ]
        for x in coordinates[0]:
            for y in coordinates[1]:
                for z in coordinates[2]:
                    position = (float(x), float(y), float(z))
                    if self._is_position_valid(position, item_dimensions, item):
                        fitness = self._calculate_position_fitness(
                            position, item_dimensions
                        )
                        if fitness > best_fitness:
                            best_fitness = fitness
                            best_position = position

        return best_position

    def _is_position_valid(
        self,
        position: Tuple[float, float, float],
        item_dims: Tuple[float, float, float],
        item: SmartItem,
    ) -> bool:
        """Check container bounds, mass capacity, and box non-overlap."""
        x, y, z = position
        container_dims = self.container.get_internal_dimensions()
        if (
            x < 0 or y < 0 or z < 0
            or x + item_dims[0] > container_dims[0] + 1e-12
            or y + item_dims[1] > container_dims[1] + 1e-12
            or z + item_dims[2] > container_dims[2] + 1e-12
        ):
            return False

        # Excluding the same object lets this feasibility check validate a stored placement.
        other_placed_items = [
            (placed_item, placed_position)
            for placed_item, placed_position in self.placed_items
            if placed_item is not item
        ]
        placed_mass = sum(
            placed_item.get_current_mass()
            for placed_item, _ in other_placed_items
        )
        if placed_mass + item.get_current_mass() > self.container.max_mass_capacity:
            return False

        for placed_item, placed_position in other_placed_items:
            placed_dims = placed_item.get_current_dimensions()
            px, py, pz = placed_position
            separated = (
                x + item_dims[0] <= px
                or px + placed_dims[0] <= x
                or y + item_dims[1] <= py
                or py + placed_dims[1] <= y
                or z + item_dims[2] <= pz
                or pz + placed_dims[2] <= z
            )
            if not separated:
                return False

        return True

    def _calculate_position_fitness(
        self,
        position: Tuple[float, float, float],
        item_dims: Tuple[float, float, float],
    ) -> float:
        """Score height, corner proximity, and relative base area."""
        x, y, z = position
        container_length, container_width, _ = self.container.get_internal_dimensions()
        gravity_fitness = 1.0 / (1.0 + z)
        corner_fitness = 1.0 / (1.0 + x + y)
        stability_fitness = (
            item_dims[0] * item_dims[1]
        ) / (container_length * container_width)
        return (
            0.4 * gravity_fitness
            + 0.3 * corner_fitness
            + 0.3 * stability_fitness
        )

    def pack_items(self) -> Dict:
        """Run the shape-preparation, sorting, and greedy placement sequence."""
        start_time = time.time()
        space_analysis = self.analyze_space_requirements()
        optimized_items = self.optimize_item_shapes()
        optimized_items.sort(
            key=lambda item: item.get_current_volume(), reverse=True
        )

        for item in optimized_items:
            position = self.find_optimal_position(item)
            if position is None:
                continue

            self.placed_items.append((item, position))
            self.unplaced_items.remove(item)
            self.optimization_history.append(
                {
                    "item_id": item.id,
                    "position": position,
                    "shape": item.current_shape,
                    "volume": item.get_current_volume(),
                    "placed_count": len(self.placed_items),
                }
            )

        total_placed_volume = sum(
            item.get_current_volume() for item, _ in self.placed_items
        )
        container_volume = self.container.get_internal_volume()
        return {
            "placed_items": len(self.placed_items),
            "unplaced_items": len(self.unplaced_items),
            "space_utilization": total_placed_volume / container_volume,
            "total_volume_placed": total_placed_volume,
            "shape_changes": self.shape_changes,
            "energy_consumed": self.energy_consumed,
            "computation_time": time.time() - start_time,
            "space_analysis": space_analysis,
        }

print("Greedy programmable-matter packing heuristic is defined.")

## 3. Container-loading instance

The following fixed instance uses a 40 ft container's approximate external dimensions and a set of mixed rectangular items. The names and dimensions are inputs to the simulation, not a claim that every item has the physical properties of a real shipment. The container's internal dimensions are calculated from the external dimensions and wall thickness. The printed initial utilization is total item volume divided by usable container volume; it is a volume ratio, not proof that the items can be packed.

In [ ]:
def create_container_loading_instance():
    """Create the fixed container and cargo instance used in this notebook."""
    container = SmartContainer(
        id=1,
        base_dimensions=(12.0, 2.35, 2.39),
        max_mass_capacity=28000.0,
        programmable=True,
        wall_thickness=0.05,
    )

    items = [
        SmartItem(id=1, base_dimensions=(1.2, 0.8, 1.5), base_mass=800, programmable=True),
        SmartItem(id=2, base_dimensions=(1.2, 0.8, 1.5), base_mass=750, programmable=True),
        SmartItem(id=3, base_dimensions=(1.0, 1.0, 1.2), base_mass=600, programmable=True),
        SmartItem(id=4, base_dimensions=(0.6, 0.4, 0.5), base_mass=50, programmable=True),
        SmartItem(id=5, base_dimensions=(0.6, 0.4, 0.5), base_mass=45, programmable=True),
        SmartItem(id=6, base_dimensions=(0.5, 0.5, 0.4), base_mass=40, programmable=False),
        SmartItem(id=7, base_dimensions=(0.5, 0.5, 0.4), base_mass=42, programmable=False),
        SmartItem(id=8, base_dimensions=(0.3, 0.2, 0.25), base_mass=15, programmable=True),
        SmartItem(id=9, base_dimensions=(0.3, 0.2, 0.25), base_mass=14, programmable=True),
        SmartItem(id=10, base_dimensions=(0.25, 0.25, 0.2), base_mass=12, programmable=True),
        SmartItem(id=11, base_dimensions=(0.25, 0.25, 0.2), base_mass=11, programmable=True),
        SmartItem(id=12, base_dimensions=(0.4, 0.3, 0.3), base_mass=20, programmable=True),
        SmartItem(id=13, base_dimensions=(0.8, 0.6, 0.7), base_mass=120, programmable=True),
        SmartItem(id=14, base_dimensions=(0.7, 0.5, 0.6), base_mass=95, programmable=True),
        SmartItem(id=15, base_dimensions=(0.4, 0.4, 0.5), base_mass=25, programmable=True),
        SmartItem(id=16, base_dimensions=(0.35, 0.35, 0.45), base_mass=22, programmable=True),
    ]
    return container, items


container, items = create_container_loading_instance()
container_dimensions = container.get_internal_dimensions()
container_volume = container.get_internal_volume()
total_item_volume = sum(item.get_current_volume() for item in items)

print(f"Container interior (L × W × H): {container_dimensions[0]:.2f} × {container_dimensions[1]:.2f} × {container_dimensions[2]:.2f} m")
print(f"Container interior volume: {container_volume:.2f} m³")
print(f"Total items: {len(items)}")
print(f"Programmable items: {sum(item.programmable for item in items)}/{len(items)}")
print(f"Total initial item volume: {total_item_volume:.2f} m³")
print(f"Initial volume ratio: {100 * total_item_volume / container_volume:.1f}%")

## 4. Run the packing heuristic

This cell executes the defined sequence and reports the heuristic's outcome. A shape-change count of zero means the volume-based rule did not request compression for this instance; it is not evidence that compression is unnecessary for every loading problem. The reported utilization is the placed item volume divided by the usable container volume. It does not measure stability, safety, or optimality.

In [ ]:
optimizer = ProgrammableMatterOptimizer(container, items)
results = optimizer.pack_items()

assert results["placed_items"] + results["unplaced_items"] == len(items)
assert all(
    optimizer._is_position_valid(
        position, item.get_current_dimensions(), item
    )
    for item, position in optimizer.placed_items
)
assert np.isclose(
    results["total_volume_placed"] / container.get_internal_volume(),
    results["space_utilization"],
)

print(f"Heuristic computation time: {results['computation_time']:.3f} s")
print(f"Items placed: {results['placed_items']}/{len(items)}")
print(f"Items not placed: {results['unplaced_items']}")
print(f"Placed-volume utilization: {100 * results['space_utilization']:.1f}%")
print(f"Volume placed: {results['total_volume_placed']:.2f} m³")
print(f"Shape changes: {results['shape_changes']}")
print(f"Illustrative energy estimate: {results['energy_consumed']:.2f} J")

analysis = results["space_analysis"]
print("\nInitial space analysis:")
print(f"  Initial volume ratio: {100 * analysis['space_utilization']:.1f}%")
print(f"  Theoretical volume reduction under allowed compression: {analysis['compression_potential']:.2f} m³")
print(f"  Volume ratio if all programmable items were compressed: {100 * analysis['optimized_utilization']:.1f}%")

print("\nPlacement details:")
for item, position in optimizer.placed_items:
    print(
        f"  Item {item.id}: {item.current_shape} at "
        f"({position[0]:.2f}, {position[1]:.2f}, {position[2]:.2f}) m"
    )
if optimizer.unplaced_items:
    print("Unplaced item IDs:", [item.id for item in optimizer.unplaced_items])

## 5. Three-dimensional placement visualization

The plot shows the container interior and the axis-aligned boxes selected by the heuristic. Colors distinguish the item's programmability and shape state; the legend defines each color. A three-dimensional perspective can make relative dimensions difficult to judge, so the geometric checks in the algorithm—not the appearance of the plot—determine feasibility.

In [ ]:
def visualize_programmable_matter_packing(container, placed_items, results):
    """Plot the interior wireframe and the placed rectangular items."""
    fig = plt.figure(figsize=(14, 9))
    ax = fig.add_subplot(111, projection="3d")
    length, width, height = container.get_internal_dimensions()

    edges = [
        ([0, length], [0, 0], [0, 0]),
        ([0, length], [width, width], [0, 0]),
        ([0, 0], [0, width], [0, 0]),
        ([length, length], [0, width], [0, 0]),
        ([0, length], [0, 0], [height, height]),
        ([0, length], [width, width], [height, height]),
        ([0, 0], [0, width], [height, height]),
        ([length, length], [0, width], [height, height]),
        ([0, 0], [0, 0], [0, height]),
        ([length, length], [0, 0], [0, height]),
        ([0, 0], [width, width], [0, height]),
        ([length, length], [width, width], [0, height]),
    ]
    for edge in edges:
        ax.plot3D(*edge, color="black", linewidth=1.5, alpha=0.7)

    colors = {
        "programmable_rectangular": "seagreen",
        "programmable_compressed": "tomato",
        "programmable_expanded": "royalblue",
        "nonprogrammable": "gray",
    }
    for item, position in placed_items:
        if not item.programmable:
            color = colors["nonprogrammable"]
        else:
            color = colors[f"programmable_{item.current_shape}"]

        x, y, z = position
        item_length, item_width, item_height = item.get_current_dimensions()
        vertices = [
            [x, y, z],
            [x + item_length, y, z],
            [x + item_length, y + item_width, z],
            [x, y + item_width, z],
            [x, y, z + item_height],
            [x + item_length, y, z + item_height],
            [x + item_length, y + item_width, z + item_height],
            [x, y + item_width, z + item_height],
        ]
        faces = [
            [vertices[0], vertices[1], vertices[2], vertices[3]],
            [vertices[4], vertices[5], vertices[6], vertices[7]],
            [vertices[0], vertices[1], vertices[5], vertices[4]],
            [vertices[2], vertices[3], vertices[7], vertices[6]],
            [vertices[0], vertices[3], vertices[7], vertices[4]],
            [vertices[1], vertices[2], vertices[6], vertices[5]],
        ]
        ax.add_collection3d(
            Poly3DCollection(
                faces, alpha=0.65, facecolor=color,
                edgecolor="black", linewidth=0.5
            )
        )

    legend_elements = [
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=colors["programmable_rectangular"], markersize=10, label="Programmable, rectangular"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=colors["programmable_compressed"], markersize=10, label="Programmable, compressed"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=colors["programmable_expanded"], markersize=10, label="Programmable, expanded"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=colors["nonprogrammable"], markersize=10, label="Non-programmable"),
    ]
    ax.set_xlabel("Length (m)")
    ax.set_ylabel("Width (m)")
    ax.set_zlabel("Height (m)")
    ax.set_title(
        "Greedy Container Loading: "
        f"{100 * results['space_utilization']:.1f}% placed-volume utilization"
    )
    ax.set_xlim(0, length)
    ax.set_ylim(0, width)
    ax.set_zlim(0, height)
    ax.legend(handles=legend_elements, loc="upper left")
    ax.view_init(elev=20, azim=45)
    plt.tight_layout()
    plt.show()


visualize_programmable_matter_packing(
    container, optimizer.placed_items, results
)

## 6. Controlled comparison: compression enabled or disabled

The comparison runs the same greedy grid-search heuristic on the same item dimensions. The comparison case disables shape changes; it is not a comparison with every traditional loading method. Both runs use fresh item objects so that a shape state from one run cannot affect the other. Computation times are environment-dependent, so they should not be interpreted as a general performance benchmark.

In [ ]:
def compare_with_traditional_methods(container, items, programmable_results):
    """Compare the same heuristic with shape changes disabled and enabled."""
    traditional_items = [
        SmartItem(
            id=item.id,
            base_dimensions=item.base_dimensions,
            base_mass=item.base_mass,
            programmable=False,
            current_shape="rectangular",
        )
        for item in items
    ]
    traditional_optimizer = ProgrammableMatterOptimizer(
        container, traditional_items
    )
    traditional_results = traditional_optimizer.pack_items()

    comparison = {
        "Method": ["Compression disabled", "Compression rule enabled"],
        "Items placed": [
            traditional_results["placed_items"],
            programmable_results["placed_items"],
        ],
        "Placed-volume utilization (%)": [
            100 * traditional_results["space_utilization"],
            100 * programmable_results["space_utilization"],
        ],
        "Volume placed (m³)": [
            traditional_results["total_volume_placed"],
            programmable_results["total_volume_placed"],
        ],
        "Shape changes": [
            traditional_results["shape_changes"],
            programmable_results["shape_changes"],
        ],
    }
    comparison_df = pd.DataFrame(comparison)
    print(comparison_df.to_string(index=False))
    print(
        "\nInterpret the table as a comparison of two settings of this heuristic, "
        "not as evidence that either setting is globally optimal."
    )
    return comparison_df, traditional_results


comparison_df, traditional_results = compare_with_traditional_methods(
    container, items, results
)

## 7. Performance dashboard

The dashboard combines the controlled utilization comparison, the shape states of placed programmable items, and cumulative placed volume over the placement sequence. Energy and computation time have different units and scales, so they are reported as text rather than combined into a misleading common bar scale.

In [ ]:
def create_comprehensive_dashboard(
    comparison_df, optimizer, results
):
    """Display placement, shape-state, progress, and metric summaries."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(14, 10)
    )

    methods = comparison_df["Method"].tolist()
    utilization = comparison_df["Placed-volume utilization (%)"].tolist()
    bars = ax1.bar(
        methods, utilization,
        color=["lightcoral", "lightgreen"], edgecolor="black"
    )
    ax1.set_ylabel("Placed-volume utilization (%)")
    ax1.set_title("Controlled heuristic comparison")
    ax1.set_ylim(0, 100)
    for bar, value in zip(bars, utilization):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{value:.1f}%",
            ha="center",
        )

    shape_counts = {"rectangular": 0, "compressed": 0, "expanded": 0}
    for item, _ in optimizer.placed_items:
        if item.programmable:
            shape_counts[item.current_shape] += 1
    nonzero_shapes = {
        shape: count for shape, count in shape_counts.items() if count > 0
    }
    if nonzero_shapes:
        ax2.pie(
            list(nonzero_shapes.values()),
            labels=list(nonzero_shapes.keys()),
            autopct="%1.0f%%",
            startangle=90,
        )
    else:
        ax2.text(0.5, 0.5, "No programmable items placed", ha="center", va="center")
    ax2.set_title("Shape states of placed programmable items")

    history = optimizer.optimization_history
    if history:
        placed_counts = [entry["placed_count"] for entry in history]
        cumulative_volumes = np.cumsum(
            [entry["volume"] for entry in history]
        )
        ax3.plot(placed_counts, cumulative_volumes, "b-o")
        ax3.set_xlabel("Items placed")
        ax3.set_ylabel("Cumulative placed volume (m³)")
        ax3.grid(True, alpha=0.3)
    else:
        ax3.text(0.5, 0.5, "No items placed", ha="center", va="center")
    ax3.set_title("Volume accumulated during placement")

    ax4.axis("off")
    metric_text = (
        f"Items placed: {results['placed_items']}\n"
        f"Items unplaced: {results['unplaced_items']}\n"
        f"Shape changes: {results['shape_changes']}\n"
        f"Illustrative energy estimate: {results['energy_consumed']:.2f} J\n"
        f"Heuristic computation time: {results['computation_time']:.3f} s"
    )
    ax4.text(0.05, 0.85, metric_text, va="top", fontsize=12)
    ax4.set_title("Run metrics (units shown individually)")

    fig.suptitle("Greedy container-loading heuristic: run summary")
    plt.tight_layout()
    plt.show()


create_comprehensive_dashboard(comparison_df, optimizer, results)

## 8. Interpreting the tier and method context

A tier label is a way to organize a discussion of computational or physical-system concepts; it is not a performance measurement. The simulation provides evidence only about the particular greedy grid search and fixed instance executed above. It does not benchmark mathematical optimization, heuristics from other sources, machine learning, digital twins, quantum computing, or physical programmable matter.

The table produced below separates capabilities represented in this notebook from capabilities that remain outside its scope. No cross-tier utilization or timing figures are inferred.

In [ ]:
def create_tier_comparison_analysis():
    """Summarize the demonstrator's scope without inventing benchmarks."""
    tier_data = {
        "Computational placement": {
            "Representation in this notebook": "Greedy search on a 0.05 m grid",
            "Evidence available": "Placements and metrics from the executed instance",
        },
        "Shape adaptation": {
            "Representation in this notebook": "Uniform compression rule for programmable items",
            "Evidence available": "Shape-change count and illustrative energy estimate",
        },
        "Physical actuation": {
            "Representation in this notebook": "Not implemented; described conceptually",
            "Evidence available": "No hardware or experimental measurements",
        },
        "Distributed control": {
            "Representation in this notebook": "Not implemented; execution is centralized",
            "Evidence available": "No distributed-system measurements",
        },
        "Cross-method performance": {
            "Representation in this notebook": "Not benchmarked",
            "Evidence available": "No external baseline data supplied",
        },
    }
    comparison = pd.DataFrame(tier_data).T
    print(comparison.to_string())
    return comparison


tier_comparison = create_tier_comparison_analysis()

## 9. Real-world implementation considerations

A simulation can clarify decisions without establishing that a technology is commercially available or technically ready. Practical deployment would require measured material properties, verified deformation limits, actuator and sensor models, safety requirements, energy measurements, and validation under real loading conditions. Costs, technology-readiness levels, availability dates, and savings are not estimated here because this instance contains no evidence for them.

The following checklist organizes the types of information an engineering feasibility study would need. It is a planning framework, not a forecast or cost-benefit result.

In [ ]:
def analyze_implementation_feasibility():
    """Return a qualitative checklist for a future feasibility study."""
    feasibility_checklist = {
        "Programmable materials": [
            "Measure achievable deformation and recovery",
            "Test durability and cargo compatibility",
            "Establish safe operating limits",
        ],
        "Actuation and sensing": [
            "Measure power and energy under representative loads",
            "Specify sensor accuracy and response time",
            "Assess fault detection and safe shutdown behavior",
        ],
        "Container and cargo integration": [
            "Validate usable dimensions and structural constraints",
            "Model support, load balance, and handling requirements",
            "Define interfaces for loading and unloading operations",
        ],
        "Operational and economic evidence": [
            "Collect measured installation and maintenance costs",
            "Compare against a defined operational baseline",
            "Evaluate reliability, safety, and lifecycle impacts",
        ],
    }
    for area, requirements in feasibility_checklist.items():
        print(f"\n{area}:")
        for requirement in requirements:
            print(f"  - {requirement}")
    return feasibility_checklist


feasibility_checklist = analyze_implementation_feasibility()

## 10. Conclusions and further study

This notebook demonstrates one transparent packing sequence: estimate whether to compress programmable items, order items by volume, enumerate candidate grid positions, test boundaries, non-overlap, and mass capacity, and choose the highest-scoring feasible position. It also visualizes the resulting placements and compares the same heuristic with its compression rule disabled.

The result is a heuristic solution for one fixed instance, not a proof of global optimality or a demonstration of physical programmable matter. The score favors low and corner positions and a relatively large base, but it does not enforce support or load balance. The grid can miss feasible placements between sampled coordinates, and the model does not consider rotation or loading sequence. A zero shape-change count should be read as the outcome of the stated rule on this instance; no sensitivity analysis is included here.

Natural extensions within this modeling framework include adding experimentally justified constraints for support, orientation, load balance, or loading order, and testing additional documented instances. Any such extension would require explicit data and validation; this notebook does not claim results for them.

In [ ]:
def generate_final_conclusions(results, comparison_df, items):
    """Print a concise summary tied to the executed instance."""
    baseline_utilization = comparison_df.loc[
        comparison_df["Method"] == "Compression disabled",
        "Placed-volume utilization (%)",
    ].iloc[0]
    enabled_utilization = comparison_df.loc[
        comparison_df["Method"] == "Compression rule enabled",
        "Placed-volume utilization (%)",
    ].iloc[0]

    print("Run summary")
    print("=" * 60)
    print(f"Items in instance: {len(items)}")
    print(f"Items placed: {results['placed_items']}")
    print(f"Items unplaced: {results['unplaced_items']}")
    print(f"Placed-volume utilization: {100 * results['space_utilization']:.1f}%")
    print(f"Compression-disabled utilization: {baseline_utilization:.1f}%")
    print(f"Compression-rule utilization: {enabled_utilization:.1f}%")
    print(f"Shape changes: {results['shape_changes']}")
    print(f"Illustrative energy estimate: {results['energy_consumed']:.2f} J")
    print(f"Heuristic computation time: {results['computation_time']:.3f} s")
    print(
        "\nThese figures describe this run of the stated heuristic. They do not "
        "establish global optimality, real-world energy performance, or physical "
        "programmable-matter capability."
    )


generate_final_conclusions(results, comparison_df, items)